<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Table of Contents:

- [Imports and Seed](#imports-and-seed)
- [Machine Learning Model Selection and Definition](#machine-learning-model-selection-and-definition)
- [Neural Network Model Selection and Definition](#neural-network-model-selection-and-definition)
- [Normal Data Modeling](#normal-data-modeling)
- [Complex Data Modeling](#complex-data-modeling)
- [Normal vs. Complex Data](#normal-vs.-complex-data)

</div>

### ORIENTATION:

This file is intended to select and define-as closely as possible-a Machine Learning (ML) and Neural Network (NN) model in order to train and evaluate the performace with the normal tabular data vs. the complex network feature engineered data.

---

### IMPORTANT NOTE:

yay

---

### SUMMARY:
- ML Selection
    - We will use *Random Forest Classifier* to maintain as much explainability and generally solve the data imbalance in the complex data
- NN Selection
    - We will create a simple *forward-pass* NN model with tensorflow and keras
- Normal Data Evaluation
    - yay
- Complex Data Evaluation
    - yay
- Normal vs. Complex Evaluation
    - yay

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Imports and Seed

- [Back to Table of Contents](#table-of-contents)

</div>

In [1]:
# Necessary for Modeling related functions
import modeling as m

import numpy as np
import pandas as pd
import networkx as nx
import igraph as ig
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight
import tensorflow as tf
from tensorflow.keras import layers, models
import joblib
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

2026-03-10 17:37:08.095895: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Set the seed in multiple areas as the default 3703
m.set_reproducibility()

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Machine Learning Model Selection and Definition

- [Back to Table of Contents](#table-of-contents)

</div>

### SUMMARY:
- We will use *Random Forest Classifier* to maintain as much explainability and generally solve the data imbalance in the complex data

---

### DISCUSSION:

Generally in the realm of Artificial Intelligence, Machine Learning (ML) models when compared with Neural Network (NN) models are primarily superior due to their explainability capabilities which is vital for deeper understanding of data and ultimately better decisionmaking.  Unfortunately, to keep things understandable for people, there's a lot of higher-level math and repetitive application of statistics observed in NNs - which is why they generally perform significantly better - that does not exist in ML models.  This is to say that the purpose of defining the best ML model for **BOTH** normal data and complex data training is primarily to observe if there's better performance with complex data training which in turn allows for more explainability of the dataset.

That being said, this dataset is a classification problem so classification models make the obvious sense to implement.  The initial assumption is that the *Decision Tree Classifier* (DTC) would be the ideal ML for comparison;  However, in `explore_complex_networks.ipynb`, we observed that the distribution of `benign` attacks became well over 99% of the data.  This data imbalance is something that DTCs fail at preventing bias and for this reason, a sort of randomness in which features to explore and how would effectively make the model blind to this data imbalance.  Thankfully, the *Random Forest Classifer* (RFC) is does exactly this.

Thus, for ML comparison of normal data and complex data, we will implement the RFC.



In [ ]:
def get_ml_model(trees:int=100) -> RandomForestClassifier:
    '''
    About
    -----
    - Creates and returns a basic sklearn RandomForestClassifier for model training

    Parameters
    ----------
    - trees (int) :
        - Default: 100
        - The number of decision trees to be created where each tree is randomly trained on a subset of the data.
          Essentially, this is like creating a voting block of whether or not something is significant during decisions

    Returns
    -------
    - RandomForestClassifier
    '''
    rfc_model = RandomForestClassifier(
        n_estimators=trees,      # This is just the number of "trees" we are creating
        class_weight='balanced', # This generally solves the imbalance issue
        max_features='sqrt',     # This generally solves the bias issue
        random_state=3703        # This is to ensure reproducibility
    )
    return rfc_model

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Neural Network Model Selection and Definition

- [Back to Table of Contents](#table-of-contents)

</div>

### SUMMARY:

- We will create a simple *forward-pass* NN model with tensorflow and keras

---

### DISCUSSION:

Inversely from the `Machine Learning Model Selection and Definition` discussion, NNs when compared with ML models are primarily superior in performance due to heavy reliance of higher-level math and repetitive statistics.  The biggest emphasis is the notion of *backpropagation* where it is best to visualize as a decision tree;  From some initial input, we incrementally make decisions until we arrive at the destination and in this case, return a prediction of what the answer is.  However, unlike decisions tree where this is the end of the process, *backpropagation* will take a wrong answer as a sign to walk backwards in all their decisions and adjust *weights and biases* for every decision until the correct answer is arrived at.  This is conducted as many times as is defined in the NN model structure with other parameters which is generally why NNs are notorious for being a *black box* in the sense of being impossible to explain their decisions and interpret them as key points of the dataset.  To emphasize, because so much higher-level math and repetitive statistical weighting and biasing, NNs lose much of their explainability, but generally performs significantly better than ML models.  Basically, it brute forces a pattern in the dataset regardless if that pattern actually means something in order to get the most correct answers as possible.

With this said, this section is to delinate NNs "brute forcing" patterns from the normal dataset and NNs "brute forcing" patterns that are mathematically created with complex network ideology from the complex dataset.  In other words, will the NN perform better without being told what underlying pattern(s) may or may not exist via the normal dataset or will the NN perform better by being told what underlying pattern(s) may or may not exist via the complex dataset.

Thus, for NN comparison, we will utilize tensorflow and keras libraries to define an **as basic of a model as possible** to prevent extreme overfitting on whatever patterns the NN thinks it sees.  Additionally, to maintain as close of comparison to the ML model selection as a *Random Forest Classifier*, we will define layers to implement a **forward-pass** concept.

In [7]:
def get_nn_model(num_features_to_train:int,
                 num_targets:int=21) -> models.Sequential:
    '''
    About
    -----
    - Creates and returns a simple forward-pass Neural Network for model training
    - This definition is to closely resemble the RandomForestClassifer as much as possible via the forward-pass

    Parameters
    ----------
    - num_features_to_train (int) :
        - The number of features being used from the dataset to train the NN on
    - num_targets (int) :
        - The number of categorical targets to predict

    Returns
    -------
    - models.Sequential
        - A tensorflow.keras NN model
    '''
    # ----- Define NN Structure -------------------------------------------------------------------
    nn_model = models.Sequential([

        # Input layer (Starting point of decision making)
        layers.Input(shape=(num_features_to_train,)),

        # Small hidden layers to prevent "brute force" memorization
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),

        # Output layer (using softmax for multiclass)
        layers.Dense(num_targets, activation='softmax') 
    ])
    
    # ----- Define Backpropagation Methodology ----------------------------------------------------
    nn_model.compile(
        optimizer='adam',                       # How weights/biases work
        loss='sparse_categorical_crossentropy', # How significant was the incorrectness
        metrics=['accuracy',                    # The metrics to optimize
                 tf.keras.metrics.SparseCategoricalAccuracy(name='cat_acc')
        ]
    )

    return nn_model

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Normal Data Modeling

- [Back to Table of Contents](#table-of-contents)

</div>

### SUMMARY:

- 

---

a

In [3]:
# Read in the PREPARED normal dataset
normal_df = pd.read_parquet('datasets/prepared_normal_short.parquet')

In [4]:
# Prepare the data for ML/NN modeling
normal_data_preparation_dict = m.prepare_data_for_training(normal_df)

In [7]:
# Conduct ML model training/testing and saving of model/metrics
normal_ml_metrics = m.train_ml(normal_data_preparation_dict)

Training Random Forest on 2659367 rows...
Model successfully saved as models/ml_normal_model.joblib!
Testing Random Forest...
Random Forest testing complete!
Obtaining performance metrics and saving...
Random Forest metrics successfully saved as metrics/ml_normal_metrics.parquet!


In [8]:
normal_ml_metrics

,target,precision,recall,f1-score,support
attack_name,,,,,
analysis,0,0.137931,0.727273,0.231884,11.000000
backdoor,1,0.738739,0.872340,0.800000,94.000000
benign,2,0.994977,0.597902,0.746947,124569.000000
bot,3,1.000000,0.998590,0.999294,709.000000
brute_force,4,0.978583,0.967427,0.972973,614.000000
ddos,5,0.993311,0.974160,0.983642,107624.000000
dos,6,0.986448,0.955099,0.970521,88484.000000
exploits,7,0.789809,0.794872,0.792332,156.000000
fuzzers,8,0.688525,0.756757,0.721030,111.000000


In [5]:
# Conduct NN model training/testing and saving of model/metrics
normal_nn_metrics = m.train_nn(normal_data_preparation_dict)

Training Neural Network on 2659367 rows...
Epoch 1/50
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.4803 - cat_acc: 0.4803 - loss: 2.2805 - val_accuracy: 0.5224 - val_cat_acc: 0.5224 - val_loss: 1.4627
Epoch 2/50
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.5303 - cat_acc: 0.5303 - loss: 1.5270 - val_accuracy: 0.5232 - val_cat_acc: 0.5232 - val_loss: 1.2983
Epoch 3/50
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.5483 - cat_acc: 0.5483 - loss: 1.2370 - val_accuracy: 0.5473 - val_cat_acc: 0.5473 - val_loss: 1.1607
Epoch 4/50
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5734 - cat_acc: 0.5734 - loss: 1.1115 - val_accuracy: 0.5667 - val_cat_acc: 0.5667 - val_loss: 1.0590
Epoch 5/50
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5989 - cat_acc: 0.5989 - loss: 0.9973 - val_accuracy: 0.6043 - val_cat_acc: 0.6043 - val_loss: 0.9840
Epoch 6/50
1299/1299 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.6270 - cat_acc: 0.6270 - loss: 0.9508 -

In [6]:
normal_nn_metrics

,target,precision,recall,f1-score,support
attack_name,,,,,
analysis,0,0.101124,0.818182,0.180000,11.000000
backdoor,1,0.863158,0.872340,0.867725,94.000000
benign,2,0.986183,0.370710,0.538861,124569.000000
bot,3,0.761290,0.998590,0.863941,709.000000
brute_force,4,0.868228,0.965798,0.914418,614.000000
ddos,5,0.987204,0.922564,0.953790,107624.000000
dos,6,0.981400,0.935615,0.957961,88484.000000
exploits,7,0.033452,0.602564,0.063385,156.000000
fuzzers,8,0.283582,0.684685,0.401055,111.000000


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Complex Data Modeling

- [Back to Table of Contents](#table-of-contents)

</div>

### SUMMARY:

- 

---

a

In [3]:
# Read in the PREPARED complex dataset
complex_df = pd.read_parquet('datasets/prepared_complex.parquet')

In [4]:
# Prepare the data for ML/NN modeling
complex_data_preparation_dict = m.prepare_data_for_training(complex_df, cols_to_drop=['attack', 'target', 'source_ip', 'destination_ip'])

In [8]:
# Conduct ML model training/testing and saving of model/metrics
complex_ml_metrics = m.train_ml(complex_data_preparation_dict,
                                'models/ml_complex_model.joblib',
                                'metrics/ml_complex_metrics.parquet')

Training Random Forest on 168963 rows...
Model successfully saved as models/ml_complex_model.joblib!
Testing Random Forest...
Random Forest testing complete!
Obtaining performance metrics and saving...
Random Forest metrics successfully saved as metrics/ml_complex_metrics.parquet!


/opt/anaconda3/envs/COMP3703/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/COMP3703/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/COMP3703/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

In [9]:
complex_ml_metrics

,target,precision,recall,f1-score,support
attack_name,,,,,
analysis,0,0.000000,0.000000,0.000000,2.000000
backdoor,1,0.000000,0.000000,0.000000,2.000000
benign,2,0.995878,0.999240,0.997556,23692.000000
bot,3,0.000000,0.000000,0.000000,1.000000
brute_force,4,1.000000,1.000000,1.000000,1.000000
ddos,5,0.428571,0.300000,0.352941,10.000000
dos,6,0.700000,0.583333,0.636364,12.000000
exploits,7,0.750000,0.750000,0.750000,4.000000
fuzzers,8,0.000000,0.000000,0.000000,4.000000


In [5]:
# Conduct NN model training/testing and saving of model/metrics
complex_nn_metrics = m.train_nn(complex_data_preparation_dict,
                                'models/nn_complex_model.keras',
                                'metrics/nn_complex_history.parquet',
                                'metrics/nn_complex_metrics.parquet')

Training Neural Network on 168963 rows...
Epoch 1/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.3897 - cat_acc: 0.3897 - loss: 4.7292 - val_accuracy: 0.6525 - val_cat_acc: 0.6525 - val_loss: 2.7180
Epoch 2/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7470 - cat_acc: 0.7470 - loss: 2.9602 - val_accuracy: 0.8225 - val_cat_acc: 0.8225 - val_loss: 2.3263
Epoch 3/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8700 - cat_acc: 0.8700 - loss: 2.5865 - val_accuracy: 0.9121 - val_cat_acc: 0.9121 - val_loss: 1.7450
Epoch 4/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9216 - cat_acc: 0.9216 - loss: 2.3937 - val_accuracy: 0.9335 - val_cat_acc: 0.9335 - val_loss: 1.2857
Epoch 5/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8925 - cat_acc: 0.8925 - loss: 2.2264 - val_accuracy: 0.8557 - val_cat_acc: 0.8557 - val_loss: 1.0924
Epoch 6/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7342 - cat_acc: 0.7342 - loss: 2.1033 - val_accuracy: 0.6731 - 

/opt/anaconda3/envs/COMP3703/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/COMP3703/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/COMP3703/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

In [8]:
complex_nn_metrics

,target,precision,recall,f1-score,support
attack_name,,,,,
analysis,0,0.000000,0.000000,0.000000,2.000000
backdoor,1,0.000000,0.000000,0.000000,2.000000
benign,2,0.999364,0.464545,0.634260,23692.000000
bot,3,0.002959,1.000000,0.005900,1.000000
brute_force,4,0.000000,0.000000,0.000000,1.000000
ddos,5,0.333333,0.200000,0.250000,10.000000
dos,6,0.333333,0.166667,0.222222,12.000000
exploits,7,1.000000,1.000000,1.000000,4.000000
fuzzers,8,0.000000,0.000000,0.000000,4.000000


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Normal vs. Complex Data

- [Back to Table of Contents](#table-of-contents)

</div>

### SUMMARY:

- 

---

a